# 02 — Splits and baselines

## Split design

**Time split on `received_dttm`.** The model will be used on *future* calls, so we evaluate it the same way: fit on earlier months and score on later ones. A random split would let the model learn from calls on the same days as the ones it is scored on (same roadworks, events, staffing), which overstates accuracy.

| split | received_dttm (inclusive) | use |
|---|---|---|
| train | 2025-01-01 → 2025-08-31 | fitting |
| valid | 2025-09-01 → 2025-09-30 | every modelling choice |
| test  | 2025-10-01 → 2025-12-31 | one final evaluation |

**Grouped by `call_number`.** Rows are unit responses, and units on the same call share the location, the time and the conditions. If one unit's row were in train and another's in valid, the model would effectively have seen the answer. The split is assigned per call (each call has a single `received_dttm`, which is asserted), and no call appears in two splits.

**Test is held out until the end.** Every decision (features, hyperparameters, thresholds) is made on valid. Looking at test along the way would quietly tune choices to it, and the final number would no longer be an honest estimate of performance on new data.

**Spatial folds (for Stage 3).** Within train, `spatial_folds` gives leave-one-battalion-out folds, to check how well the model generalises to an area it has not seen.

Cleaning rules R1–R7 (including the R6/R7 travel-time bounds) are applied identically to every split, so all metrics below are on cleaned data.

All logic lives in `src/`; this notebook only calls it.

In [1]:
import sys
from pathlib import Path

# The notebook runs from notebooks/; make the project root importable.
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src import baselines
from src.splits import spatial_folds

pd.set_option("display.width", 200)

In [2]:
r = baselines.run()  # loads data/processed/ambulance_clean.parquet; fits on train, predicts valid
r["split_counts"]

,rows,calls
split,,
train,66943,66358
valid,8266,8195
test,25252,25027


## Spatial folds (train only, used in Stage 3)

In [3]:
train = r["df"][r["df"]["split"] == "train"]
folds = spatial_folds(train)

train rows per battalion:
battalion
B02    13853
B03    12751
B04     7162
B01     5778
B10     5501
B09     5251
B08     5226
B06     4273
B05     4057
B07     3091
no battalion has < 500 rows; nothing merged

leave-one-battalion-out: 10 folds
fold           0      1     2     3     4     5     6     7     8     9
battalion                                                              
B01            0      0     0  5778     0     0     0     0     0     0
B02        13853      0     0     0     0     0     0     0     0     0
B03            0  12751     0     0     0     0     0     0     0     0
B04            0      0  7162     0     0     0     0     0     0     0
B05            0      0     0     0     0     0     0     0  4057     0
B06            0      0     0     0     0     0     0  4273     0     0
B07            0      0     0     0     0     0     0     0     0  3091
B08            0      0     0     0     0     0  5226     0     0     0
B09            0      0     0     0

## Baselines on VALID

All baselines are medians fitted on train (the median is the MAE-optimal constant).

- **B0**: global median
- **B1**: median by `original_priority_cat`
- **B2**: median by `station_area` × `original_priority_cat` (full category)

A cell with fewer than 30 training rows falls back to the next coarser level: B2 → B1 → B0.

*Revised in Stage 3.* B2 originally used the 3-level priority group (3 / 2 / other). Collapsing the letter codes merged fast (E) and slow (A) codes and made B2 worse than B1 on the "other" group, so B2 now uses the full category. B3 (B2 × hour band 00–06 / 07–18 / 19–23) was dropped: the hour band added nothing on valid (MAE 237.5 for both B2 and B3 in Stage 2).

In [4]:
r["fallbacks"]  # which level produced each valid prediction

,B2,B1,B0
B0 global median,0,0,8266
B1 priority_cat,0,8261,5
B2 station x priority_cat,8018,243,5


In [5]:
baselines.metric_table(r["valid"]["travel_time_s"], r["preds"])

n    MAE   RMSE  MedAE  P90AE  within±120s_%
segment baseline                                                                  
ALL     B0 global median           8266  281.5  408.8  200.0  640.0           30.8
        B1 priority_cat            8266  240.3  357.4  165.0  510.0           38.3
        B2 station x priority_cat  8266  232.6  352.3  154.0  511.0           41.4

In [6]:
baselines.metric_table(r["valid"]["travel_time_s"], r["preds"], r["valid"]["unit_type"])

n    MAE   RMSE  MedAE  P90AE  within±120s_%
segment baseline                                                                  
MEDIC   B0 global median           6318  285.2  412.3  199.0  659.0           30.7
        B1 priority_cat            6318  242.4  359.1  166.0  525.3           38.7
        B2 station x priority_cat  6318  235.5  353.5  156.0  519.3           41.1
PRIVATE B0 global median           1948  269.5  397.1  203.0  570.3           31.2
        B1 priority_cat            1948  233.4  351.7  162.0  471.3           37.0
        B2 station x priority_cat  1948  223.1  348.1  147.8  461.6           42.2

In [7]:
baselines.metric_table(r["valid"]["travel_time_s"], r["preds"], r["valid"]["priority_group"])

n    MAE   RMSE  MedAE  P90AE  within±120s_%
segment baseline                                                                  
2       B0 global median           2981  372.5  510.4  273.0  804.0           24.0
        B1 priority_cat            2981  307.2  416.6  249.0  589.0           24.6
        B2 station x priority_cat  2981  299.7  410.6  236.5  599.5           26.3
3       B0 global median           3564  189.5  267.1  157.0  325.7           38.0
        B1 priority_cat            3564  163.7  267.6  112.0  318.0           53.5
        B2 station x priority_cat  3564  153.8  260.0   99.0  295.0           58.6
other   B0 global median           1721  314.7  451.1  220.0  725.0           27.8
        B1 priority_cat            1721  282.8  405.6  208.0  594.0           30.6
        B2 station x priority_cat  1721  279.5  404.9  199.0  598.0           31.8

## B2 80% interval on VALID (train 10th–90th percentile per cell)

In [8]:
baselines.interval_table(r["valid"], r["lo"], r["hi"])

,coverage_%,mean_width_s
segment,,
ALL,79.6,689.4
unit_type=MEDIC,79.9,686.7
unit_type=PRIVATE,78.8,697.9
priority_group=2,79.1,901.0
priority_group=3,80.5,446.1
priority_group=other,78.6,826.4
